# Face embedding training (MobileFaceNet + ArcFace) on Colab

This notebook mirrors the local training pipeline in the `Facial-Recognition`
repo, adapted to run on a Colab T4 GPU instead of the local AMD iGPU.

**Manual steps required from you:**
1. Runtime -> Change runtime type -> T4 GPU
2. Have a Kaggle API token ready (kaggle.com -> Settings -> API -> Create
   New Token) -- you'll be asked to type your username/key into two masked
   prompts below (nothing is uploaded or shared outside this notebook).
3. You'll be asked to authorize Google Drive access (used to persist
   checkpoints across sessions, since free Colab sessions are time-limited
   and can disconnect).

Everything else runs automatically once you execute the cells in order.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected -- set Runtime > Change runtime type > T4 GPU, then re-run.")


## 1. Install extra dependencies

(torch/torchvision/numpy/pillow/pandas ship preinstalled on Colab.)

In [ ]:
!pip install -q kaggle pyyaml


## 2. Kaggle API credentials

From kaggle.com -> Settings -> API -> Create New Token. Kaggle may show
this as a single token string (`export KAGGLE_API_TOKEN=...`) or, under
"Legacy API Credentials", as a username/key pair or downloaded
`kaggle.json` -- any of these work with the CLI. This cell handles the
newer single-token format: paste just the token value (not the
`export KAGGLE_API_TOKEN=` part) into the masked prompt. If you only have
the legacy username/key pair instead, tell Claude and it'll swap this cell.


In [ ]:
import getpass, os

os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token: ")
print("Kaggle token set.")


## 3. Download the dataset

Same source as the local setup: https://www.kaggle.com/datasets/yakhyokhuja/webface-112x112

In [ ]:
!kaggle datasets download -d yakhyokhuja/webface-112x112 -p /content --unzip
!ls /content/webface_112x112 | head -5
!ls /content/webface_112x112 | wc -l


## 4. Mount Google Drive

Checkpoints are written to Drive so they survive a session disconnect --
free Colab sessions are capped (commonly ~12h, can disconnect earlier) and
the full baseline run will likely span multiple sessions. See the "Resuming
after a disconnect" section at the end of this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/facial-recognition-checkpoints"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("checkpoints will be saved to:", CHECKPOINT_DIR)


## 5. Write the training source files

These are exact copies of the files in `src/` and `scripts/` in the local
repo (`Facial-Recognition`), kept in sync manually -- if you change the
local versions, re-generate this notebook (ask Claude) rather than editing
these cells by hand, so the two don't drift apart.


In [ ]:
%%writefile src/models/__init__.py


In [ ]:
%%writefile src/models/mobilefacenet.py
"""MobileFaceNet architecture, recreated from the original paper:

Chen et al., "MobileFaceNets: Efficient CNNs for Accurate Real-Time Face
Verification on Mobile Devices" (2018), https://arxiv.org/abs/1804.07573

Weights are randomly initialized; no pretrained checkpoints are loaded here.
"""

from __future__ import annotations

import torch
from torch import nn


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1, use_act=True):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel, stride, padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.PReLU(out_c) if use_act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class DepthwiseBlock(nn.Module):
    """Depthwise conv followed by pointwise conv (standard MobileNet separable conv)."""

    def __init__(self, in_c, out_c, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
        super().__init__()
        self.depthwise = ConvBlock(in_c, in_c, kernel, stride, padding, groups=in_c)
        self.pointwise = ConvBlock(in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0))

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class Bottleneck(nn.Module):
    """Inverted residual block (expand -> depthwise -> project), as used in MobileNetV2/MobileFaceNet."""

    def __init__(self, in_c, out_c, stride, expansion):
        super().__init__()
        self.use_residual = stride == 1 and in_c == out_c
        hidden_dim = in_c * expansion

        self.expand = ConvBlock(in_c, hidden_dim, kernel=(1, 1), stride=(1, 1), padding=(0, 0))
        self.depthwise = ConvBlock(
            hidden_dim, hidden_dim, kernel=(3, 3), stride=(stride, stride), padding=(1, 1), groups=hidden_dim
        )
        self.project = ConvBlock(hidden_dim, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), use_act=False)

    def forward(self, x):
        out = self.project(self.depthwise(self.expand(x)))
        if self.use_residual:
            out = out + x
        return out


class BottleneckStage(nn.Module):
    """A stack of `n` bottleneck blocks; only the first block uses `stride`."""

    def __init__(self, in_c, out_c, stride, expansion, n):
        super().__init__()
        layers = [Bottleneck(in_c, out_c, stride, expansion)]
        for _ in range(n - 1):
            layers.append(Bottleneck(out_c, out_c, 1, expansion))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)


class MobileFaceNet(nn.Module):
    """MobileFaceNet backbone producing a fixed-size face embedding.

    Input: 3x112x112 face crop.
    Output: L2-normalized embedding of dimension `embedding_dim`.

    Architecture (channels/strides/expansion/repeats) follows Table 1 of the
    MobileFaceNets paper, sized for 112x112 input.
    """

    # (out_channels, stride, expansion, num_blocks)
    _STAGE_CFG = [
        (64, 2, 2, 5),
        (128, 2, 4, 1),
        (128, 1, 2, 6),
        (128, 2, 4, 1),
        (128, 1, 2, 2),
    ]

    def __init__(self, embedding_dim: int = 256, input_size: int = 112):
        super().__init__()
        if input_size % 16 != 0:
            raise ValueError("input_size must be divisible by 16")

        self.stem = ConvBlock(3, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1))
        self.dw_stem = DepthwiseBlock(64, 64, kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        stages = []
        in_c = 64
        for out_c, stride, expansion, n in self._STAGE_CFG:
            stages.append(BottleneckStage(in_c, out_c, stride, expansion, n))
            in_c = out_c
        self.stages = nn.Sequential(*stages)

        self.conv_1x1 = ConvBlock(in_c, 512, kernel=(1, 1), stride=(1, 1), padding=(0, 0))

        # Global depthwise conv (GDConv) replaces global average pooling, per the paper,
        # since faces are spatially aligned and different regions carry unequal information.
        feat_map_size = input_size // 16
        self.gdconv = nn.Conv2d(512, 512, kernel_size=feat_map_size, groups=512, bias=False)
        self.gdconv_bn = nn.BatchNorm2d(512)

        self.linear = nn.Conv2d(512, embedding_dim, kernel_size=1, stride=1, padding=0, bias=False)
        self.linear_bn = nn.BatchNorm1d(embedding_dim)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu")
            elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.PReLU):
                nn.init.constant_(m.weight, 0.25)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.dw_stem(x)
        x = self.stages(x)
        x = self.conv_1x1(x)
        x = self.gdconv_bn(self.gdconv(x))
        x = self.linear(x)
        x = x.flatten(1)
        x = self.linear_bn(x)
        embedding = nn.functional.normalize(x, p=2, dim=1)
        return embedding


if __name__ == "__main__":
    model = MobileFaceNet(embedding_dim=256)
    dummy = torch.randn(2, 3, 112, 112)
    out = model(dummy)
    print("output shape:", out.shape)
    print("output norm (should be ~1.0 per row):", out.norm(dim=1))
    n_params = sum(p.numel() for p in model.parameters())
    print(f"parameters: {n_params:,}")


In [ ]:
%%writefile src/losses/__init__.py


In [ ]:
%%writefile src/losses/arcface.py
"""ArcFace additive angular margin loss head.

Deng et al., "ArcFace: Additive Angular Margin Loss for Deep Face
Recognition" (2019), https://arxiv.org/abs/1801.07698

This module is a *training-time classification head* on top of the face
embedding. It is not part of the inference-time embedding model: at
inference, only the backbone (e.g. MobileFaceNet) is used, and identities
are compared via cosine similarity between embeddings.
"""

from __future__ import annotations

import math

import torch
from torch import nn
import torch.nn.functional as F


class ArcMarginHead(nn.Module):
    """Computes ArcFace logits from L2-normalized embeddings and class labels.

    Args:
        embedding_dim: dimensionality of the input face embeddings.
        num_classes: number of identities in the training set.
        margin: additive angular margin `m` (radians), typically 0.5.
        scale: logit scale `s`, typically 64.
    """

    def __init__(self, embedding_dim: int, num_classes: int, margin: float = 0.5, scale: float = 64.0):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        self.margin = margin
        self.scale = scale

        self.weight = nn.Parameter(torch.empty(num_classes, embedding_dim))
        nn.init.xavier_normal_(self.weight)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        # Threshold beyond which cos(theta + m) would increase instead of decrease
        # (i.e. theta + m > pi); used for the numerically-stable "easy margin" fallback.
        self.threshold = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        normalized_weight = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, normalized_weight)  # (B, num_classes), already in [-1, 1]

        sine = torch.sqrt((1.0 - cosine.pow(2)).clamp(min=0.0, max=1.0))
        phi = cosine * self.cos_m - sine * self.sin_m  # cos(theta + m)

        # Where theta + m would exceed pi, fall back to a linear penalty so the
        # loss stays monotonically decreasing (Deng et al., Sec. 3.3).
        phi = torch.where(cosine > self.threshold, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        logits = logits * self.scale
        return logits


if __name__ == "__main__":
    torch.manual_seed(0)
    embed_dim, num_classes, batch = 256, 1000, 8
    head = ArcMarginHead(embed_dim, num_classes)
    embeddings = F.normalize(torch.randn(batch, embed_dim), p=2, dim=1)
    labels = torch.randint(0, num_classes, (batch,))
    logits = head(embeddings, labels)
    loss = F.cross_entropy(logits, labels)
    print("logits shape:", logits.shape)
    print("loss:", loss.item())


In [ ]:
%%writefile src/data/__init__.py


In [ ]:
%%writefile src/data/dataset.py
"""PyTorch Dataset for the CASIA-WebFace identity-disjoint split.

Reads the manifests produced by scripts/make_identity_split.py
(image_path,label CSVs). Images are already aligned to a fixed size (112x112
for the Kaggle webface-112x112 mirror -- see docs/DATASET.md), so no face
detection/alignment happens here, only normalization and light augmentation.
"""

from __future__ import annotations

import csv
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


def build_transform(image_size: int, train: bool, horizontal_flip: bool) -> transforms.Compose:
    ops = []
    if image_size != 112:
        ops.append(transforms.Resize((image_size, image_size)))
    if train and horizontal_flip:
        ops.append(transforms.RandomHorizontalFlip(p=0.5))
    ops.append(transforms.ToTensor())  # -> [0, 1]
    ops.append(transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]))  # -> [-1, 1]
    return transforms.Compose(ops)


class FaceDataset(Dataset):
    """Loads (image, integer label) pairs from a train.csv/val_seen.csv manifest."""

    def __init__(self, manifest_csv: str | Path, transform: transforms.Compose):
        self.transform = transform
        self.samples: list[tuple[str, int]] = []
        with open(manifest_csv, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append((row["image_path"], int(row["label"])))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
            img = self.transform(img)
        return img, label

    @property
    def num_classes(self) -> int:
        return max(label for _, label in self.samples) + 1


In [ ]:
%%writefile src/train.py
"""Baseline training loop: MobileFaceNet + ArcFace on the CASIA-WebFace
identity-disjoint train split (see docs/DATASET.md, scripts/make_identity_split.py).

Usage:
    python src/train.py --config configs/baseline.yaml
    python src/train.py --config configs/baseline.yaml --epochs 1 --max-steps 20  # smoke test
"""

from __future__ import annotations

import argparse
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader

from data.dataset import FaceDataset, build_transform
from losses.arcface import ArcMarginHead
from models.mobilefacenet import MobileFaceNet


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_optimizer(cfg: dict, params):
    train_cfg = cfg["train"]
    if train_cfg["optimizer"] != "sgd":
        raise ValueError(f"unsupported optimizer: {train_cfg['optimizer']}")
    return torch.optim.SGD(
        params,
        lr=train_cfg["lr"],
        momentum=train_cfg["momentum"],
        weight_decay=train_cfg["weight_decay"],
    )


def build_scheduler(cfg: dict, optimizer):
    train_cfg = cfg["train"]
    if train_cfg["lr_schedule"] != "multistep":
        raise ValueError(f"unsupported lr_schedule: {train_cfg['lr_schedule']}")
    return torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=train_cfg["lr_milestones"], gamma=train_cfg["lr_gamma"]
    )


def save_checkpoint(path: Path, epoch: int, model, head, optimizer, scheduler, cfg: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "head_state": head.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "config": cfg,
        },
        path,
    )


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, default=Path("configs/baseline.yaml"))
    parser.add_argument("--data-root", type=str, default=None, help="override data.root")
    parser.add_argument("--splits-dir", type=str, default=None, help="override data.splits_dir")
    parser.add_argument("--checkpoint-dir", type=str, default=None, help="override checkpointing.dir")
    parser.add_argument("--epochs", type=int, default=None, help="override train.epochs")
    parser.add_argument("--max-steps", type=int, default=None, help="stop after N steps per epoch (smoke test)")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--resume", type=Path, default=None, help="resume from a checkpoint saved by this script")
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    if args.data_root:
        cfg["data"]["root"] = args.data_root
    if args.splits_dir:
        cfg["data"]["splits_dir"] = args.splits_dir
    if args.checkpoint_dir:
        cfg["checkpointing"]["dir"] = args.checkpoint_dir
    if args.epochs:
        cfg["train"]["epochs"] = args.epochs

    set_seed(cfg["seed"])
    device = args.device
    print(f"device: {device}", flush=True)
    if device == "cuda":
        print(f"gpu: {torch.cuda.get_device_name(0)}", flush=True)

    splits_dir = Path(cfg["data"]["splits_dir"])
    transform = build_transform(
        cfg["data"]["image_size"], train=True, horizontal_flip=cfg["augmentation"]["horizontal_flip"]
    )
    train_ds = FaceDataset(splits_dir / "train.csv", transform)
    num_classes = train_ds.num_classes
    print(f"train images: {len(train_ds)}, identities: {num_classes}", flush=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["train"]["batch_size"],
        shuffle=True,
        num_workers=cfg["train"]["num_workers"],
        pin_memory=(device == "cuda"),
        drop_last=True,
    )

    model = MobileFaceNet(embedding_dim=cfg["model"]["embedding_dim"]).to(device)
    head = ArcMarginHead(
        cfg["model"]["embedding_dim"], num_classes, margin=cfg["arcface"]["margin"], scale=cfg["arcface"]["scale"]
    ).to(device)

    optimizer = build_optimizer(cfg, list(model.parameters()) + list(head.parameters()))
    scheduler = build_scheduler(cfg, optimizer)
    use_amp = cfg["train"]["amp"] and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch = 0
    if args.resume:
        print(f"resuming from {args.resume}", flush=True)
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        head.load_state_dict(ckpt["head_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        print(f"resumed at epoch {start_epoch}", flush=True)

    ckpt_dir = Path(cfg["checkpointing"]["dir"])
    best_loss = float("inf")

    for epoch in range(start_epoch, cfg["train"]["epochs"]):
        model.train()
        head.train()
        epoch_loss = 0.0
        n_batches = 0
        t_epoch0 = time.time()

        for step, (images, labels) in enumerate(train_loader):
            if args.max_steps is not None and step >= args.max_steps:
                break
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp, dtype=torch.float16):
                embeddings = model(images)
                logits = head(embeddings, labels)
                loss = F.cross_entropy(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            n_batches += 1

            if step % 50 == 0:
                elapsed = time.time() - t_epoch0
                imgs_per_sec = (step + 1) * cfg["train"]["batch_size"] / max(elapsed, 1e-6)
                print(
                    f"epoch {epoch} step {step}/{len(train_loader)} "
                    f"loss {loss.item():.4f} ({imgs_per_sec:.1f} img/s)",
                    flush=True,
                )

        scheduler.step()
        mean_loss = epoch_loss / max(n_batches, 1)
        print(f"epoch {epoch} done, mean_loss={mean_loss:.4f}, time={time.time()-t_epoch0:.1f}s", flush=True)

        if (epoch + 1) % cfg["checkpointing"]["save_every_epochs"] == 0:
            save_checkpoint(ckpt_dir / f"epoch_{epoch}.pt", epoch, model, head, optimizer, scheduler, cfg)

        if cfg["checkpointing"]["keep_best"] and mean_loss < best_loss:
            best_loss = mean_loss
            save_checkpoint(ckpt_dir / "best.pt", epoch, model, head, optimizer, scheduler, cfg)

    save_checkpoint(ckpt_dir / "final.pt", cfg["train"]["epochs"] - 1, model, head, optimizer, scheduler, cfg)
    print("training complete", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/make_identity_split.py
"""Build an identity-disjoint train/val/test split for CASIA-WebFace.

Expects the dataset root to contain one subdirectory per identity, each
holding that identity's face images (the standard CASIA-WebFace layout).
This has not yet been verified against the actual downloaded dataset -- run
`scripts/inspect_dataset.py` first and adjust this script if the real
layout differs.

Produces three manifests (CSV: image_path,identity_label) so that identity
labels stay consistent (0..num_train_identities-1) for the ArcFace head:

  - train.csv               images used for training
  - val_seen.csv             held-out images of TRAINING identities
                              ("seen identity, unseen image" eval)
  - test_unseen.csv          all images of identities NEVER used in training
                              ("unseen identity" eval)

The split is fully determined by `--seed`, so it is reproducible.
"""

from __future__ import annotations

import argparse
import csv
import random
from pathlib import Path

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


def discover_identities(root: Path) -> dict[str, list[Path]]:
    identities: dict[str, list[Path]] = {}
    for identity_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        images = sorted(
            p for p in identity_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if images:
            identities[identity_dir.name] = images
    return identities


def build_split(
    identities: dict[str, list[Path]],
    unseen_identity_fraction: float,
    held_out_image_fraction: float,
    seed: int,
):
    rng = random.Random(seed)

    identity_names = list(identities.keys())
    rng.shuffle(identity_names)

    n_unseen = max(1, round(len(identity_names) * unseen_identity_fraction))
    unseen_identities = set(identity_names[:n_unseen])
    train_eligible_identities = identity_names[n_unseen:]

    train_rows = []
    val_seen_rows = []
    test_unseen_rows = []

    # Training identity labels are assigned only over train-eligible
    # identities, in a fixed (shuffled) order, so label ids are stable
    # given the same seed.
    label_of = {name: idx for idx, name in enumerate(train_eligible_identities)}

    for name in train_eligible_identities:
        images = list(identities[name])
        rng.shuffle(images)
        n_held_out = max(1, round(len(images) * held_out_image_fraction)) if len(images) > 1 else 0
        held_out, kept = images[:n_held_out], images[n_held_out:]
        if not kept:
            # Never leave an identity with zero training images.
            kept, held_out = held_out, []
        label = label_of[name]
        for img in kept:
            train_rows.append((str(img), label))
        for img in held_out:
            val_seen_rows.append((str(img), label))

    for name in unseen_identities:
        for img in identities[name]:
            # Unseen identities have no training label; use the identity
            # name itself so verification pairs can be formed per-identity.
            test_unseen_rows.append((str(img), name))

    return train_rows, val_seen_rows, test_unseen_rows, label_of


def write_csv(path: Path, rows, header):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, required=True, help="CASIA-WebFace root (identity-per-folder)")
    parser.add_argument("--output-dir", type=Path, default=Path("data/splits"))
    parser.add_argument("--unseen-identity-fraction", type=float, default=0.05)
    parser.add_argument("--held-out-image-fraction", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    identities = discover_identities(args.data_root)
    if not identities:
        raise SystemExit(f"No identity subdirectories with images found under {args.data_root}")

    train_rows, val_seen_rows, test_unseen_rows, label_of = build_split(
        identities, args.unseen_identity_fraction, args.held_out_image_fraction, args.seed
    )

    write_csv(args.output_dir / "train.csv", train_rows, ["image_path", "label"])
    write_csv(args.output_dir / "val_seen.csv", val_seen_rows, ["image_path", "label"])
    write_csv(args.output_dir / "test_unseen.csv", test_unseen_rows, ["image_path", "identity_name"])
    write_csv(
        args.output_dir / "label_map.csv",
        sorted(label_of.items(), key=lambda kv: kv[1]),
        ["identity_name", "label"],
    )

    print(f"identities total:        {len(identities)}")
    print(f"train identities:        {len(label_of)}")
    print(f"unseen identities:       {len(identities) - len(label_of)}")
    print(f"train images:            {len(train_rows)}")
    print(f"val_seen images:         {len(val_seen_rows)}")
    print(f"test_unseen images:      {len(test_unseen_rows)}")
    print(f"seed:                    {args.seed}")
    print(f"wrote manifests to:      {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/baseline.yaml
# Baseline training configuration.
# Every hyperparameter that affects reproducibility should live here rather
# than as a hardcoded constant in training code.

seed: 42

data:
  root: null # path to CASIA-WebFace root (identity-per-folder layout); set locally, never committed
  splits_dir: data/splits # output of scripts/make_identity_split.py
  image_size: 112
  # Identities held out entirely (never seen during training) for the
  # "unseen identity" generalization evaluation.
  unseen_identity_fraction: 0.05
  # Within the *remaining* (training-eligible) identities, fraction of each
  # identity's images held out for the "seen identity, unseen image" eval.
  held_out_image_fraction: 0.1
  split_seed: 42

model:
  architecture: mobilefacenet
  embedding_dim: 256

arcface:
  margin: 0.5
  scale: 64.0

train:
  batch_size: 128
  epochs: 30
  optimizer: sgd
  lr: 0.1
  momentum: 0.9
  weight_decay: 5.0e-4
  lr_schedule: multistep
  lr_milestones: [16, 24, 28]
  lr_gamma: 0.1
  num_workers: 4
  amp: true # mixed precision; helps throughput on the integrated GPU

augmentation:
  horizontal_flip: true
  # Kept deliberately minimal for the baseline; extend only if validation
  # results indicate under/overfitting.

eval:
  batch_size: 256
  verification_pairs_seen: 6000
  verification_pairs_unseen: 6000
  far_targets: [0.01, 0.001]

checkpointing:
  dir: checkpoints/baseline
  save_every_epochs: 1
  keep_best: true


## 6. Build a Colab-specific config

Same hyperparameters as the local `configs/baseline.yaml` -- only the
dataset path, split output path, and checkpoint directory change (pointed
at this Colab runtime's paths and Drive).


In [ ]:
import yaml

with open("configs/baseline.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["root"] = "/content/webface_112x112"
cfg["data"]["splits_dir"] = "/content/data/splits"
cfg["checkpointing"]["dir"] = CHECKPOINT_DIR

with open("configs/baseline_colab.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(yaml.safe_dump(cfg, sort_keys=False))


## 7. Build the identity-disjoint split

Same seed (42) as local, so the split is the same partition of identities
(though images-per-identity load order may differ slightly if the Kaggle
download itself changed -- unlikely, but worth knowing).


In [ ]:
!python scripts/make_identity_split.py \
    --data-root /content/webface_112x112 \
    --output-dir /content/data/splits \
    --seed 42


## 8. Train

This runs the exact same `train.py` used locally. Expect an epoch to take
minutes rather than the ~2 hours/epoch measured on the local AMD iGPU,
given the T4's mature CUDA/cuDNN stack and higher raw FP16 throughput --
but this hasn't actually been measured yet on Colab, so treat the first
run's logged img/s as the real number, not the estimate from the local
session.


In [ ]:
%cd src
!python train.py --config ../configs/baseline_colab.yaml
%cd ..


## Resuming after a disconnect

Free Colab sessions can disconnect before training finishes. To resume:

1. Re-run cells 1-7 above (GPU check, deps, Kaggle download, Drive mount,
   source files, config, split -- these are all fast/idempotent).
2. Find the latest checkpoint under `CHECKPOINT_DIR` (printed below).
3. Run the training cell with `--resume <path-to-latest-checkpoint>` added.


In [ ]:
import glob, os

ckpts = sorted(
    glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.pt")),
    key=lambda p: int(p.split("epoch_")[-1].split(".pt")[0]),
)
print("latest checkpoint:", ckpts[-1] if ckpts else "none found yet")


In [ ]:
# Example resume command -- edit the --resume path to the checkpoint printed above, then run:
# %cd src
# !python train.py --config ../configs/baseline_colab.yaml --resume /content/drive/MyDrive/facial-recognition-checkpoints/epoch_4.pt
# %cd ..
